In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("UrbanFlow - Feature Engineering") \
    .getOrCreate()

RAW_PATH = "gs://trip-data-project-e6d9c504-e176-49d1-acd/urbanflow/raw/hvfhv/"

df = spark.read.parquet(RAW_PATH)

df = df.filter(
    (F.col("trip_miles") > 0) &
    (F.col("trip_time") > 0) &
    (F.col("pickup_datetime") <= F.col("dropoff_datetime"))
)

print("Registros después de filtros base:", f"{df.count():,}")

In [ ]:
df_features = (
    df

    # Variables temporales
    .withColumn("year", F.year("pickup_datetime"))
    .withColumn("month", F.month("pickup_datetime"))
    .withColumn("day", F.dayofmonth("pickup_datetime"))
    .withColumn("day_of_week", F.dayofweek("pickup_datetime"))
    .withColumn("pickup_hour", F.hour("pickup_datetime"))

    # Duración
    .withColumn(
        "trip_duration_minutes",
        F.col("trip_time") / 60.0
    )

    # Tiempo de espera válido
    .withColumn(
        "wait_time_minutes",
        F.when(
            F.col("request_datetime") <= F.col("pickup_datetime"),
            (
                F.unix_timestamp("pickup_datetime") -
                F.unix_timestamp("request_datetime")
            ) / 60.0
        )
    )

    # Revenue por milla
    .withColumn(
        "revenue_per_mile",
        F.when(
            F.col("trip_miles") > 0,
            F.col("base_passenger_fare") / F.col("trip_miles")
        )
    )

    # Revenue por minuto
    .withColumn(
        "revenue_per_minute",
        F.when(
            F.col("trip_time") > 0,
            F.col("base_passenger_fare") / (F.col("trip_time") / 60.0)
        )
    )

    # Velocidad promedio
    .withColumn(
        "avg_speed_mph",
        F.when(
            F.col("trip_time") > 0,
            F.col("trip_miles") / (F.col("trip_time") / 3600.0)
        )
    )

    # Relación driver pay / fare
    .withColumn(
        "driver_pay_ratio",
        F.when(
            F.col("base_passenger_fare") != 0,
            F.col("driver_pay") / F.col("base_passenger_fare")
        )
    )

    # Flags
    .withColumn(
        "flag_request_after_pickup",
        F.col("request_datetime") > F.col("pickup_datetime")
    )

    .withColumn(
        "flag_negative_fare",
        F.col("base_passenger_fare") < 0
    )

    .withColumn(
        "flag_negative_driver_pay",
        F.col("driver_pay") < 0
    )
)

print("Columnas finales:", len(df_features.columns))

df_features.select(
    "pickup_datetime",
    "year",
    "month",
    "day_of_week",
    "pickup_hour",
    "trip_duration_minutes",
    "wait_time_minutes",
    "revenue_per_mile",
    "revenue_per_minute",
    "avg_speed_mph",
    "driver_pay_ratio"
).show(10, truncate=False)

In [ ]:
CURATED_PATH = "gs://trip-data-project-e6d9c504-e176-49d1-acd/urbanflow/curated/hvfhv/"

(
    df_features
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(CURATED_PATH)
)

print("CURATED escrito correctamente en:")
print(CURATED_PATH)

print("\nValidación:")
print("Registros CURATED esperados: 710,451,024")
print("Particionado por: year, month")